In [1]:
from train_gpt2 import GPT2 
import torch
import tiktoken

/Users/atharvjain/code/GPT-2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
VOCAB_SIZE = 50257
N_EMBED = 768
N_HEAD = 12
N_LAYER = 12
CONTEXT_LEN = 1024

In [3]:
my_model = GPT2.from_pretrained()
# my_model = GPT2(vocab_size=VOCAB_SIZE, n_embed=N_EMBED, n_head=N_HEAD, n_layer=N_LAYER, context_len=CONTEXT_LEN)

my_model.eval()

device = "mps" if torch.backends.mps.is_available() else "cpu"
my_model.to(device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2489.36it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x TransformerBlock(
        (attn): Attention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
enc = tiktoken.get_encoding("gpt2")

assert enc.decode(enc.encode("Hello world")) == "Hello world" 

In [5]:
phrase = "Hello, I am a language model" 
tokens = enc.encode(phrase) 

input_ids = torch.tensor(tokens, device=device)
# input_ids

In [6]:
torch.manual_seed(42)
torch.mps.manual_seed(42)
max_length = 30 

current = input_ids.unsqueeze(0).expand(5, -1)

with torch.no_grad():
    while current.shape[1] < max_length:
        # print(current.shape)
        output = my_model(current)
        next_token_logits = output[:, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)
        topk_probs, topk_indices = torch.topk(next_token_probs, k=50) 
        next_token = torch.multinomial(topk_probs, num_samples=1)
        xcol = torch.gather(topk_indices, -1, next_token)
        current = torch.cat((current, xcol.squeeze(0)), dim=1)

In [7]:
for i in range(current.shape[0]):
    print(f"Generated sequence {i}: {enc.decode(current[i].tolist())}")

Generated sequence 0: Hello, I am a language model developer. I write lots of Python scripts using Python. I also write many Python games in parallel with Python and Python
Generated sequence 1: Hello, I am a language model. My life in your life is different. [She pauses for a moment but says: "Ah!"]


Generated sequence 2: Hello, I am a language model, not a programming language. The name of an object is a string and object-to-string operations can be
Generated sequence 3: Hello, I am a language model. This is so very important to me. My language is as simple as I can imagine it to be. If
Generated sequence 4: Hello, I am a language model programming program. I take advantage of my language knowledge to generate program that is as simple, easy, and extensible
